# Phase 1: Data Cleaning

In [ ]:
import pandas as pd

df = pd.read_csv("raw_2021.csv", header=None)


header_row_index = df[df.apply(lambda row: row.astype(str).str.contains("STATE/UT NAME", case=False).any(), axis=1)].index[0]

df.columns = df.iloc[header_row_index]
df = df[header_row_index + 1:].reset_index(drop=True)


df = df.dropna(how="all")


df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace("/", "_", regex=False)
      .str.replace(".", "", regex=False)
      .str.replace("%", "pct", regex=False)
      .str.replace(" ", "_", regex=False)
)

for col in ["state_ut_name", "ac_name", "candidate_name"]:
    df[col] = df[col].str.upper().str.strip()


numeric_cols = ["age", "general", "postal", "total", "pct_votes_polled", "total_electors"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")


df[['candidate_no', 'candidate_name_clean']] = df['candidate_name'].str.extract(r'(\d+)\s+(.*)')

df = df.drop(columns=['candidate_name'])
df = df.rename(columns={'candidate_name_clean': 'candidate_name'})


print(df.head())
print(df.columns.tolist())


In [ ]:
df.to_csv("clean_2021.csv", index=False)


# Phase 2: Constituency Results

In [ ]:
import pandas as pd

# Load the cleaned election data
df = pd.read_csv("clean_2021.csv")

key_cols = [
    'state_ut_name', 'ac_no', 'ac_name', 'candidate_no', 'candidate_name',
    'party', 'category', 'sex', 'age', 'total', 'pct_votes_polled', 'total_electors'
]


df = df[key_cols]


df_sorted = df.sort_values(['state_ut_name', 'ac_no', 'total'], ascending=[True, True, False])


def get_winner_runner_up(group):
    winner = group.iloc[0]
    runner_up = group.iloc[1] if len(group) > 1 else None
    margin = winner['total'] - runner_up['total'] if runner_up is not None else winner['total']
    
    return pd.Series({
        'winner_name': winner['candidate_name'],
        'winner_party': winner['party'],
        'winner_votes': winner['total'],
        'runner_up_name': runner_up['candidate_name'] if runner_up is not None else None,
        'runner_up_party': runner_up['party'] if runner_up is not None else None,
        'runner_up_votes': runner_up['total'] if runner_up is not None else None,
        'vote_margin': margin,
        'total_votes_polled': group['total'].sum(),
        'num_candidates': len(group)
    })


constituency_results = df_sorted.groupby(
    ['state_ut_name', 'ac_no', 'ac_name'],
    group_keys=False
).apply(get_winner_runner_up).reset_index()

constituency_results['winner_vote_share_pct'] = constituency_results['winner_votes'] / constituency_results['total_votes_polled'] * 100
constituency_results['runner_up_vote_share_pct'] = constituency_results['runner_up_votes'] / constituency_results['total_votes_polled'] * 100

constituency_results['vote_margin_pct'] = constituency_results['vote_margin'] / constituency_results['total_votes_polled'] * 100


constituency_results = constituency_results.sort_values(['state_ut_name', 'ac_no']).reset_index(drop=True)

constituency_results.to_csv("constituency_results_2021.csv", index=False)


print(constituency_results.head())
print(constituency_results.columns.tolist())


# Phase 3: Party & State Aggregation

In [ ]:

import pandas as pd

constituency_results = pd.read_csv("constituency_results_2021.csv")

party_summary = constituency_results.groupby('winner_party').agg(
    seats_won=('winner_name', 'count'),
    total_votes_won=('winner_votes', 'sum'),
    total_votes_polled=('total_votes_polled', 'sum')
).reset_index()

party_summary['vote_share_pct'] = party_summary['total_votes_won'] / party_summary['total_votes_polled'] * 100


party_summary = party_summary.sort_values('seats_won', ascending=False).reset_index(drop=True)


state_party_summary = constituency_results.groupby(['state_ut_name', 'winner_party']).agg(
    seats_won=('winner_name', 'count'),
    total_votes_won=('winner_votes', 'sum'),
    total_votes_polled=('total_votes_polled', 'sum')
).reset_index()


state_party_summary['vote_share_pct'] = state_party_summary['total_votes_won'] / state_party_summary['total_votes_polled'] * 100


state_party_summary = state_party_summary.sort_values(['state_ut_name', 'seats_won'], ascending=[True, False]).reset_index(drop=True)


runner_up_summary = constituency_results.groupby(['state_ut_name', 'runner_up_party']).agg(
    seats_runner_up=('runner_up_name', 'count')
).reset_index()


state_party_summary = pd.merge(
    state_party_summary,
    runner_up_summary,
    left_on=['state_ut_name', 'winner_party'],
    right_on=['state_ut_name', 'runner_up_party'],
    how='left'
).drop(columns=['runner_up_party'])

state_party_summary['seats_runner_up'] = state_party_summary['seats_runner_up'].fillna(0).astype(int)

party_summary.to_csv("party_summary_2021.csv", index=False)
state_party_summary.to_csv("state_party_summary_2021.csv", index=False)

print("All-India Party Summary:")
print(party_summary.head())
print("\nState-wise Party Summary:")
print(state_party_summary.head())


# Phase 4: Analysis & Visualization

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os


output_dir = "phase4_outputs"
os.makedirs(output_dir, exist_ok=True)

constituency_results = pd.read_csv("constituency_results_2021.csv")
party_summary = pd.read_csv("party_summary_2021.csv")
state_party_summary = pd.read_csv("state_party_summary_2021.csv")

plt.figure(figsize=(10,6))
sns.barplot(data=party_summary, x='winner_party', y='seats_won', palette='tab10')
plt.title('Seats Won by Party - 2021 (All India)')
plt.xlabel('Party')
plt.ylabel('Seats Won')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "all_india_seats.png"))
plt.close()

plt.figure(figsize=(10,6))
sns.barplot(data=party_summary, x='winner_party', y='vote_share_pct', palette='tab20')
plt.title('Vote Share by Party - 2021 (All India)')
plt.xlabel('Party')
plt.ylabel('Vote Share (%)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "all_india_vote_share.png"))
plt.close()


plt.figure(figsize=(12,8))
sns.barplot(data=state_party_summary, x='state_ut_name', y='seats_won', hue='winner_party', dodge=True)
plt.title('State-wise Seats Won by Party - 2021')
plt.xlabel('State/UT')
plt.ylabel('Seats Won')
plt.xticks(rotation=45)
plt.legend(title='Party', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "statewise_seats.png"))
plt.close()

plt.figure(figsize=(12,8))
sns.barplot(data=state_party_summary, x='state_ut_name', y='vote_share_pct', hue='winner_party', dodge=True)
plt.title('State-wise Vote Share by Party - 2021')
plt.xlabel('State/UT')
plt.ylabel('Vote Share (%)')
plt.xticks(rotation=45)
plt.legend(title='Party', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "statewise_vote_share.png"))
plt.close()


plt.figure(figsize=(10,6))
sns.histplot(constituency_results['vote_margin_pct'], bins=30, kde=True)
plt.title('Distribution of Vote Margin (%) Across Constituencies')
plt.xlabel('Vote Margin (%)')
plt.ylabel('Number of Constituencies')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "vote_margin_distribution.png"))
plt.close()


top10_close = constituency_results.nsmallest(10, 'vote_margin_pct')[[
    'state_ut_name','ac_name','winner_name','winner_party','runner_up_name','runner_up_party','vote_margin_pct'
]]
top10_close.to_csv(os.path.join(output_dir, "top10_closest_contests.csv"), index=False)

top10_large = constituency_results.nlargest(10, 'vote_margin_pct')[[
    'state_ut_name','ac_name','winner_name','winner_party','runner_up_name','runner_up_party','vote_margin_pct'
]]
top10_large.to_csv(os.path.join(output_dir, "top10_largest_margins.csv"), index=False)

party_summary.to_csv(os.path.join(output_dir, "party_summary.csv"), index=False)
state_party_summary.to_csv(os.path.join(output_dir, "state_party_summary.csv"), index=False)

print(f"Phase 4 outputs saved to folder: {output_dir}")


# Phase 5: Modelling

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
# Load dataset
df = pd.read_csv("constituency_results_2021.csv")


features = ['total_votes_polled', 'num_candidates', 'vote_margin', 'winner_vote_share_pct', 'runner_up_vote_share_pct']
X = df[features]


party_counts = df['winner_party'].value_counts()
df_filtered = df[df['winner_party'].isin(party_counts[party_counts>=2].index)]
X_filtered = df_filtered[features]


le_party = LabelEncoder()
y_filtered = le_party.fit_transform(df_filtered['winner_party'])


X_train, X_test, y_train, y_test = train_test_split(X_filtered, y_filtered, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)


y_pred = rf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

labels_in_test = np.unique(y_test)
print("\nClassification Report:\n", classification_report(
    y_test,
    y_pred,
    labels=labels_in_test,
    target_names=le_party.inverse_transform(labels_in_test),
    zero_division=0
))


df_filtered['predicted_party_encoded'] = rf.predict(X_filtered)
df_filtered['predicted_party'] = le_party.inverse_transform(df_filtered['predicted_party_encoded'])


df_filtered.to_csv("constituency_predictions_2021_filtered.csv", index=False)


sns.set_style("whitegrid")

feat_importance = pd.DataFrame({
    'feature': features,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(8,5))
sns.barplot(x='importance', y='feature', data=feat_importance, palette='viridis')
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig("feature_importance.png")
plt.show()

party_counts_pred = df_filtered['predicted_party'].value_counts().reset_index()
party_counts_pred.columns = ['party', 'num_constituencies']

plt.figure(figsize=(10,6))
sns.barplot(x='party', y='num_constituencies', data=party_counts_pred, palette='tab10')
plt.title("Predicted Party Distribution (2021 Filtered Constituencies)")
plt.ylabel("Number of Constituencies")
plt.xlabel("Party")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("predicted_party_distribution.png")
plt.show()

cm = confusion_matrix(y_test, y_pred, labels=labels_in_test)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=le_party.inverse_transform(labels_in_test), 
            yticklabels=le_party.inverse_transform(labels_in_test), cmap='Blues')
plt.title("Confusion Matrix")
plt.xlabel("Predicted Party")
plt.ylabel("Actual Party")
plt.tight_layout()
plt.savefig("confusion_matrix.png")
plt.show()


party_counts_actual = df_filtered['winner_party'].value_counts().head(10)
party_counts_pred_top = df_filtered['predicted_party'].value_counts().head(10)

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.pie(party_counts_actual, labels=party_counts_actual.index, autopct='%1.1f%%', startangle=140)
plt.title("Actual Party Distribution (Top 10)")

plt.subplot(1,2,2)
plt.pie(party_counts_pred_top, labels=party_counts_pred_top.index, autopct='%1.1f%%', startangle=140)
plt.title("Predicted Party Distribution (Top 10)")

plt.tight_layout()
plt.savefig("party_distribution_comparison.png")
plt.show()


# Phase 6: Projection

In [ ]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import os

# Create output directory
os.makedirs("outputs_2026", exist_ok=True)

df = pd.read_csv("constituency_results_2021.csv")

# Fill missing vote shares if any
df['winner_vote_share_pct'] = df['winner_vote_share_pct'].fillna(0)
df['runner_up_vote_share_pct'] = df['runner_up_vote_share_pct'].fillna(0)
df['vote_margin_pct'] = df['vote_margin'] / df['total_votes_polled'] * 100

feature_cols = ['winner_vote_share_pct', 'runner_up_vote_share_pct', 'vote_margin_pct', 'num_candidates']
X = df[feature_cols]

le_party = LabelEncoder()
y = le_party.fit_transform(df['winner_party'])

# Train Random Forest on 2021 data
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X, y)


np.random.seed(42)
# Random swing per constituency (-5% to +5%)
swing_pct = np.random.normal(0, 3, size=len(df))
X_2026 = X.copy()
X_2026['winner_vote_share_pct'] = np.clip(X_2026['winner_vote_share_pct'] + swing_pct, 0, 100)


df['predicted_party_encoded'] = rf.predict(X_2026)
df['predicted_party_2026'] = le_party.inverse_transform(df['predicted_party_encoded'])





# By State
state_seat_tally_2026 = df.groupby(['state_ut_name', 'predicted_party_2026']).size().reset_index(name='projected_seats_2026')
state_seat_tally_2026 = state_seat_tally_2026.sort_values(['state_ut_name', 'projected_seats_2026'], ascending=[True, False])
state_seat_tally_2026.to_csv("outputs_2026/state_seat_tally_2026.csv", index=False)

# Constituency-level
df.to_csv("outputs_2026/constituency_projection_2026.csv", index=False)


sns.set_style("whitegrid")


#State-wise Top Parties - Heatmap
state_pivot = state_seat_tally_2026.pivot(index='state_ut_name', columns='predicted_party_2026', values='projected_seats_2026').fillna(0)
plt.figure(figsize=(12,8))
sns.heatmap(state_pivot, annot=True, fmt='.0f', cmap='YlGnBu')
plt.title("Projected State-wise Seat Distribution 2026")
plt.ylabel("State/UT")
plt.xlabel("Party")
plt.tight_layout()
plt.savefig("outputs_2026/state_seat_heatmap_2026.png")
plt.show()

# 5c: Party Vote Share Distribution (Histogram)
plt.figure(figsize=(10,6))
sns.histplot(df['winner_vote_share_pct'], bins=20, kde=True, color='skyblue')
plt.title("Distribution of Winner Vote Share % (2021 Base)")
plt.xlabel("Vote Share %")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig("outputs_2026/winner_vote_share_distribution.png")
plt.show()

# =========================
# Step 6: Summary Outputs
# =========================
print("=== Sample 2026 Projection (first 5 constituencies) ===")
print(df[['state_ut_name', 'ac_name', 'predicted_party_2026', 'winner_party', 'winner_vote_share_pct']].head())

print("\n=== Projected State-wise Seat Tally 2026 (first 10 rows) ===")
print(state_seat_tally_2026.head(10))
